In [0]:
spark.catalog.setCurrentCatalog("databricks_project")
spark.catalog.setCurrentDatabase("gold_schema")
display(spark.catalog.currentCatalog(), spark.catalog.currentDatabase())

In [0]:
gold_dim_album = spark.read.table("databricks_project.silver_schema.dim_album")
gold_dim_artist = spark.read.table("databricks_project.silver_schema.dim_artist")
gold_dim_genre = spark.read.table("databricks_project.silver_schema.dim_genre")
gold_dim_mediatype = spark.read.table("databricks_project.silver_schema.dim_mediatype")
gold_dim_track = spark.read.table("databricks_project.silver_schema.dim_track")
gold_dim_customer = spark.read.table("databricks_project.silver_schema.dim_customer")
gold_dim_employee = spark.read.table("databricks_project.silver_schema.dim_employee")
gold_dim_invoice = spark.read.table("databricks_project.silver_schema.dim_invoice")
gold_fact_inv_line = spark.read.table("databricks_project.silver_schema.fact_inv_line")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, current_date, date_sub

gold_agg_fact_sales = (
    gold_fact_inv_line.alias("f")
    .join(gold_dim_track.alias("t"), F.col("f.Trk_sk") == F.col("t.Track_sk"), "left")
    .join(gold_dim_invoice.alias("i"), F.col("f.Inv_sk") == F.col("i.Invoice_sk"), "left")
    .join(gold_dim_customer.alias("c"), F.col("f.Cust_sk") == F.col("c.Customer_sk"), "left")
    .join(gold_dim_employee.alias("e"), F.col("f.Emp_sk") == F.col("e.Employee_sk"), "left")
    .select(
        F.col("f.InvoiceLineId").alias("InvoiceLineId"),
        F.col("f.Inv_sk").alias("Invoice_sk"),
        F.col("f.Trk_sk").alias("Track_sk"),
        # prefer common name fields; fall back to safer alternatives if missing
        F.col("t.Name").alias("TrackName"),
        F.col("t.AlbumId").alias("AlbumId"),
        F.col("t.GenreId").alias("GenreId"),
        F.col("f.Cust_sk").alias("Customer_sk"),
        F.concat_ws(" ", F.col("c.FirstName"), F.col("c.LastName")).alias("CustomerName"),
        F.col("f.Emp_sk").alias("Employee_sk"),
        F.concat_ws(" ", F.col("e.FirstName"), F.col("e.LastName")).alias("EmployeeName"),
        F.coalesce(F.col("i.InvoiceDate"), F.col("f.InvoiceDate")).alias("InvoiceDate"),
        F.col("f.UnitPrice").alias("UnitPrice"),
        F.col("f.Quantity").alias("Quantity"),
        F.round(F.col("f.UnitPrice") * F.col("f.Quantity"), 2).alias("Revenue")
    )
)

gold_agg_sales_track = (
    gold_agg_fact_sales
    .groupBy("Track_sk", "TrackName", "AlbumId", "GenreId",F.year("InvoiceDate").alias("InvoiceDate"))
    .agg(
        F.sum("Revenue").alias("TotalSales"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.countDistinct("Invoice_sk").alias("DistinctInvoices"),
        F.round(F.expr("sum(Revenue)/nullif(sum(Quantity),0)"), 2).alias("AvgPricePerUnit")
    )
    .orderBy(F.desc("TotalSales"))
)

gold_agg_sales_customer = (
    gold_agg_fact_sales
    .groupBy("Customer_sk", "CustomerName",F.year("InvoiceDate").alias("InvoiceDate"))
    .agg(
        F.sum("Revenue").alias("TotalSales"),
        F.countDistinct("Invoice_sk").alias("TotalOrders"),
        F.round(F.expr("sum(Revenue)/nullif(count(DISTINCT Invoice_sk),0)"), 2).alias("AvgOrderValue"),
        F.countDistinct("Track_sk").alias("DistinctTracksBought"),
        F.min("InvoiceDate").alias("FirstOrderDate"),
        F.max("InvoiceDate").alias("LastOrderDate")
    )
    .orderBy(F.desc("TotalSales"))
)

gold_agg_sales_employee = (
    gold_agg_fact_sales
    .groupBy("Employee_sk", "EmployeeName",F.year("InvoiceDate").alias("InvoiceDate"))
    .agg(
        F.sum("Revenue").alias("TotalSales"),
        F.countDistinct("Invoice_sk").alias("InvoicesHandled"),
        F.countDistinct("Customer_sk").alias("CustomersServed"),
        F.round(F.expr("sum(Revenue)/nullif(count(DISTINCT Invoice_sk),0)"), 2).alias("AvgSalesPerInvoice")
    )
    .orderBy(F.desc("TotalSales"))
)



In [0]:
display(gold_agg_fact_sales)
display(gold_agg_sales_track)
display(gold_agg_sales_customer)
display(gold_agg_sales_employee)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Aggregate sales by album from the reporting-level fact (gold_agg_fact_sales)
gold_agg_sales_album = (
    gold_agg_fact_sales
    .groupBy("AlbumId", F.year("InvoiceDate").alias("InvoiceDate"))
    .agg(
        F.sum("Revenue").cast(T.FloatType()).alias("TotalSales"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.countDistinct("Track_sk").alias("TracksCount"),
        F.countDistinct("Invoice_sk").alias("DistinctInvoices"),
        F.round(F.expr("sum(Revenue)/nullif(sum(Quantity),0)"), 2).alias("AvgPricePerUnit"),
        F.round(F.expr("sum(Revenue)/nullif(count(DISTINCT Invoice_sk),0)"), 2).alias("AvgSalesPerInvoice")
    )
    .join(gold_dim_album.select("AlbumId", "Title", "ArtistId"), on="AlbumId", how="left")
    .join(gold_dim_artist.select("ArtistId", "Name").alias("artist"), on="ArtistId", how="left")
    .select(
        "AlbumId",
        F.coalesce(F.col("Title"), F.concat(F.lit("Album_"), F.col("AlbumId"))).alias("AlbumTitle"),
        "ArtistId",
        F.coalesce(F.col("artist.Name"), F.concat(F.lit("Artist_"), F.col("ArtistId"))).alias("ArtistName"),
        "InvoiceDate",
        "TracksCount",
        "TotalSales",
        "TotalQuantity",
        "DistinctInvoices",
        "AvgPricePerUnit",
        "AvgSalesPerInvoice"
    )
    .orderBy(F.desc("TotalSales"))
)

# sales by genre

gold_agg_sales_genre = (
    gold_agg_fact_sales
    .groupBy("GenreId", F.year("InvoiceDate").alias("InvoiceDate"))
    .agg(
        F.sum("Revenue").cast(T.DoubleType()).alias("TotalSales"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.countDistinct("Invoice_sk").alias("DistinctInvoices"),
        F.countDistinct("Track_sk").alias("DistinctTracks"),
        F.round(F.expr("sum(Revenue)/nullif(sum(Quantity),0)"), 2).alias("AvgPricePerUnit"),
        F.round(F.expr("sum(Revenue)/nullif(count(DISTINCT Invoice_sk),0)"), 2).alias("AvgSalesPerInvoice")
    )
    .join(gold_dim_genre.select("GenreId", "Name"), on="GenreId", how="left")
    .select(
        "GenreId",
        F.coalesce(F.col("Name"), F.lit("Unknown")).alias("GenreName"),
        "DistinctTracks",
        "InvoiceDate",
        "TotalSales",
        "TotalQuantity",
        "DistinctInvoices",
        "AvgPricePerUnit",
        "AvgSalesPerInvoice"
    )
    .orderBy(F.desc("TotalSales"))
)

# sales by composer

gold_agg_sales_composer = (
    gold_agg_fact_sales
    .join(gold_dim_track.select("Track_sk", "Composer"), on="Track_sk", how="left")
    .withColumn(
        "Composer",
        F.when(F.col("Composer").isNull() | (F.trim(F.col("Composer")) == ""), F.lit("Unknown"))
         .otherwise(F.col("Composer"))
    )
    .groupBy("Composer")
    .agg(
        F.sum("Revenue").cast(T.DoubleType()).alias("TotalSales"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.countDistinct("Invoice_sk").alias("DistinctInvoices"),
        F.countDistinct("Track_sk").alias("DistinctTracksSold"),
        F.round(F.expr("sum(Revenue)/nullif(sum(Quantity),0)"), 2).alias("AvgPricePerUnit")
    )
    .orderBy(F.desc("TotalSales"))
)

gold_agg_sales_composer = gold_agg_sales_composer.filter(F.when(F.col("Composer")!="Unknown", True))

In [0]:

display(gold_agg_sales_genre)

In [0]:
gold_dim_album.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_album")
                                                   
gold_dim_artist.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_artist")

gold_dim_customer.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_customer")

gold_dim_employee.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_employee")

gold_dim_genre.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_genre")

gold_dim_invoice.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_invoice")

gold_dim_mediatype.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_mediatype")

gold_dim_track.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_dim_track")

gold_fact_inv_line.write.mode("overwrite").saveAsTable("databricks_project.gold_schema.gold_fact_inv_line")

gold_agg_fact_sales.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_fact_sales")

gold_agg_sales_track.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_track")

gold_agg_sales_album.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_album")

gold_agg_sales_customer.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_customer")

gold_agg_sales_employee.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_employee")

gold_agg_sales_genre.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_genre")

gold_agg_sales_composer.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("databricks_project.gold_schema.gold_agg_sales_composer")